In [1]:
import contextlib
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import plotly.figure_factory as ff
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
import wandb
from tqdm.autonotebook import tqdm

from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    LossConstraintConfig,
)
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.experiments.multimodal_gaussian.canonical import (
    get_canonical_chart_transport_configs,
    get_canonical_chart_transport_monitor_configs,
)
from src.experiments.multimodal_gaussian.config import MultimodalGaussianTrainingConfig
from src.experiments.multimodal_gaussian.state import MultimodalTrainingRuntime

/tmp/ipykernel_2233201/587479404.py:11: TqdmExperimentalWarning:

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)



In [2]:
num_modes = 8
latent_dimension = 64
ambient_dimension = 64
experiment_name = f"{datetime.now():%m%d%H%M%S}-{uuid4().hex[:6]}"
wandb_project = "diffusive-latent-generation"

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=0.1,
    offset=0.0,
    ambient_dimension=ambient_dimension,
)

training_config = MultimodalGaussianTrainingConfig.initialize(
    seed=0,
    train_batch_size=4096,
    eval_batch_size=4096,
    integrated_n_steps=1_000_000,
    chart_transport_config=get_canonical_chart_transport_configs(
        data_config=data_config,
        latent_dimension=latent_dimension,
    ),
    monitor_config=get_canonical_chart_transport_monitor_configs(),
    folder=(
        Path("/home/nlyu/Code/diffusive-latent-generation/artifacts/multimodal_gaussian")
        / experiment_name
    ),
)

training_config.visualize()

In [3]:
cuda_device = 0
runtime = MultimodalTrainingRuntime.initialize(
    tc=training_config,
    cuda_device=cuda_device,
    wandb_project=wandb_project,
    run_name=experiment_name,
)

Using bfloat16 Automatic Mixed Precision (AMP)
Seed set to 0
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/nlyu/.netrc.
wandb: Currently logged in as: lyuxingjian (lyuxingjian-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Pretrain

Chart pretrain is now delegated to the runtime. It logs constraint and conditioning monitors on the configured schedule.


In [4]:
latest_pretrain_metrics = runtime.chart_pretrain()
latest_pretrain_metrics

pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

[pretrain] step 1/1000: train: chart_loss=0.6353, data_cycle_loss=0.0685, prior_cycle_loss=0.5665, zero_mean_loss=0.0003, latent_norm_loss=3.0795; monitor: constraint_reconstruction_mean=4.0406, constraint_reconstruction_max=4.7289, constraint_latent_norm_mean=6.0462, encoder_conditioning_mean=6.7445, encoder_conditioning_max=10.5842, decoder_conditioning_mean=1.1160, decoder_conditioning_max=1.1784
[pretrain] step 1000/1000: train: chart_loss=0.0020, data_cycle_loss=0.0006, prior_cycle_loss=0.0012, zero_mean_loss=0.0001, latent_norm_loss=2.1896; monitor: constraint_reconstruction_mean=0.2456, constraint_reconstruction_max=0.3029, constraint_latent_norm_mean=2.1078, encoder_conditioning_mean=4.8995, encoder_conditioning_max=8.1432, decoder_conditioning_mean=1.0908, decoder_conditioning_max=1.1625


{'train_chart_loss': 0.001996976090595126,
 'train_data_cycle_loss': 0.000573044060729444,
 'train_prior_cycle_loss': 0.001204970758408308,
 'train_zero_mean_loss': 6.513483822345734e-05,
 'train_latent_norm_loss': 2.1896133422851562,
 'monitor_constraint_reconstruction_mean': 0.24560366570949554,
 'monitor_constraint_reconstruction_max': 0.3028714954853058,
 'monitor_constraint_latent_norm_mean': 2.1077589988708496,
 'monitor_encoder_conditioning_mean': 4.899483680725098,
 'monitor_encoder_conditioning_max': 8.143211364746094,
 'monitor_decoder_conditioning_mean': 1.0907971858978271,
 'monitor_decoder_conditioning_max': 1.1625468730926514}

## Train noise critic

In [5]:
latest_critic_pretrain_metrics = runtime.critic_pretrain()
latest_critic_pretrain_metrics

critic_pretrain:   0%|          | 0/1000 [00:00<?, ?it/s]

[critic_pretrain] step 1/1000: train: critic_loss=1.1339; monitor: critic_monitor_snapshot_score_norm_mean=195.4803, critic_monitor_transport_norm_mean=35.2465, critic_monitor_transport_norm_max=39.5675, critic_monitor_transport_t_min=0.0100, critic_monitor_transport_t_max=0.9900
[critic_pretrain] step 1000/1000: train: critic_loss=0.0176; monitor: critic_monitor_snapshot_score_norm_mean=349.5923, critic_monitor_transport_norm_mean=15.3518, critic_monitor_transport_norm_max=19.3407, critic_monitor_transport_t_min=0.0100, critic_monitor_transport_t_max=0.9900


{'train_critic_loss': 0.017581457272171974,
 'monitor_critic_monitor_snapshot_score_norm_mean': 349.5923156738281,
 'monitor_critic_monitor_transport_norm_mean': 15.351786613464355,
 'monitor_critic_monitor_transport_norm_max': 19.34073257446289,
 'monitor_critic_monitor_transport_t_min': 0.009999999776482582,
 'monitor_critic_monitor_transport_t_max': 0.9900000095367432}

## Integrated training

Integrated training is now delegated to the runtime. Each integrated step performs one chart-repair update, `n_critic_updates_every_transport_step` critic updates, then one chart-transport update. The integrated monitor stack logs constraint, critic, conditioning, and sampling monitors on the configured schedule.


In [ ]:
latest_integrated_metrics = runtime.integrated()
latest_integrated_metrics

integrated:   0%|          | 0/1000000 [00:00<?, ?it/s]

[integrated] step 1/1000000: train: critic=0.0542, repair=0.0017, data_cycle=0.0005, prior_cycle=0.0012, data_dual=0.0000, prior_dual=0.0000, transport=0.0009, enc_transport=0.0000, dec_transport=0.0008, field=9.7714; monitor: recon_err=0.4177, latent_norm=2.1306, score=253.3298, field=27.3547, enc_cond=4.9344, dec_cond=1.1049, kl_0p001=4.5527, kl_0p003=4.3841, kl_0p01=4.3414, kl_0p03=4.3291
[integrated] step 1000/1000000: train: critic=0.0944, repair=0.0017, data_cycle=0.0005, prior_cycle=0.0012, data_dual=0.0000, prior_dual=0.0000, transport=0.0010, enc_transport=0.0000, dec_transport=0.0009, field=7.8403; monitor: recon_err=0.2253, latent_norm=1.9870, score=177.4025, field=21.8207, enc_cond=7.0930, dec_cond=1.0209, kl_0p001=4.2333, kl_0p003=4.0985, kl_0p01=4.0658, kl_0p03=4.0597
[integrated] step 2000/1000000: train: critic=0.1184, repair=0.0005, data_cycle=0.0001, prior_cycle=0.0004, data_dual=0.0000, prior_dual=0.0000, transport=0.0002, enc_transport=0.0000, dec_transport=0.0002, 